# A4 — Local Binary Patterns + Histogram of Oriented Gradients

**Goal:** Visualise LBP micro-texture patterns and HOG brushstroke orientations on Manet images. Combine them and check whether PCA separates the two artists.

**Key papers:**
- Ojala, Pietikäinen & Mäenpää (2002), *IEEE T-PAMI* 24(7) — LBP
- Dalal & Triggs (2005), CVPR — HOG

**Why these together:** LBP captures **micro-texture** (paint surface, fine grain, impasto) and is invariant to monotonic intensity changes. HOG captures **macro brushstroke edges** and their orientations. Together they cover complementary spatial scales.

---

### Mathematics

**LBP** at radius $R$, $P$ sample points around pixel $(x,y)$ with central value $g_c$:
$$\text{LBP}_{P,R}(x,y) = \sum_{i=0}^{P-1} s(g_i - g_c) \cdot 2^i, \qquad s(x) = \begin{cases}1 & x \geq 0 \\ 0 & x < 0\end{cases}$$

**Rotation-invariant uniform LBP** ($\text{LBP}^{riu2}_{P,R}$): only patterns with $\leq 2$ binary transitions (0→1 or 1→0). For $P=8$: 58 uniform patterns + 1 non-uniform = **59 distinct codes**.

**HOG** per cell of $s \times s$ pixels:
- Gradient: $|\nabla I|$ and $\theta = \arctan(\partial_y I / \partial_x I)$
- Histogram: 9 unsigned orientation bins (0–180°), vote by magnitude
- Block normalisation: $\mathbf{v}_{\text{norm}} = \mathbf{v} / \sqrt{\|\mathbf{v}\|^2 + \varepsilon}$

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from notebooks.research.utils import load_image, to_gray, show_pair
from skimage.feature import local_binary_pattern, hog
from skimage.color import rgb2gray
from sklearn.decomposition import PCA

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
PATH_MANET        = Path('../../data/manet/manet_example.tif')
PATH_CONTEMPORARY = Path('../../data/contemporary/contemporary_example.tif')
LBP_P = 8     # Number of sample points
LBP_R = 1     # Radius
HOG_CELL = 16 # Cell size in pixels
# ─────────────────────────────────────────────────────────────────────────────

img_m  = load_image(PATH_MANET)
img_c  = load_image(PATH_CONTEMPORARY)
gray_m = to_gray(img_m)
gray_c = to_gray(img_c)
show_pair(img_m, img_c)

---
## 1. LBP Image — What Micro-Textures Dominate?

The LBP-encoded image assigns a code (0–58 for uniform LBP) to each pixel. The spatial distribution of codes reveals the micro-texture structure.

In [ ]:
lbp_m = local_binary_pattern(gray_m, P=LBP_P, R=LBP_R, method='uniform')
lbp_c = local_binary_pattern(gray_c, P=LBP_P, R=LBP_R, method='uniform')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LBP micro-texture images  (LBP⁸₁ uniform — 59 codes)', fontsize=12)

for col, (name, img, lbp) in enumerate([
    ('Manet', img_m, lbp_m), ('Contemporary', img_c, lbp_c)
]):
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'{name}')
    axes[0, col].axis('off')

    im = axes[1, col].imshow(lbp, cmap='nipy_spectral', vmin=0, vmax=58)
    plt.colorbar(im, ax=axes[1, col], fraction=0.03, label='LBP code')
    axes[1, col].set_title(f'{name} — LBP pattern map')
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

---
## 2. LBP Histogram — Compare Manet vs Contemporary

In [ ]:
def lbp_histogram(lbp_image: np.ndarray, n_bins: int = 59) -> np.ndarray:
    """Normalised LBP histogram."""
    hist, _ = np.histogram(lbp_image.ravel(), bins=n_bins, range=(0, n_bins))
    return hist.astype(float) / (hist.sum() + 1e-9)


hist_m = lbp_histogram(lbp_m)
hist_c = lbp_histogram(lbp_c)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

x = np.arange(59)
w = 0.4
axes[0].bar(x - w/2, hist_m, w, color='#2196F3', alpha=0.8, label='Manet')
axes[0].bar(x + w/2, hist_c, w, color='#FF5722', alpha=0.8, label='Contemporary')
axes[0].set_xlabel('LBP^{riu2} code (0–58)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('LBP histogram: Manet vs Contemporary\nBins 0–57: uniform patterns | Bin 58: non-uniform')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Difference
diff = hist_m - hist_c
axes[1].bar(x, diff, color=np.where(diff > 0, '#2196F3', '#FF5722'), alpha=0.8)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xlabel('LBP code')
axes[1].set_ylabel('Manet − Contemporary')
axes[1].set_title('Histogram difference  (blue = Manet higher, orange = Contemporary higher)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Chi-squared distance between histograms
chi2 = 0.5 * np.sum((hist_m - hist_c)**2 / (hist_m + hist_c + 1e-9))
print(f'Chi-squared distance between LBP histograms: {chi2:.6f}')
print('(Higher = more different texture)')

---
## 3. HOG — Brushstroke Orientation Visualisation

In [ ]:
# Compute HOG with visualisation
hog_feat_m, hog_vis_m = hog(
    gray_m,
    orientations=9,
    pixels_per_cell=(HOG_CELL, HOG_CELL),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    visualize=True,
    feature_vector=True
)

hog_feat_c, hog_vis_c = hog(
    gray_c,
    orientations=9,
    pixels_per_cell=(HOG_CELL, HOG_CELL),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    visualize=True,
    feature_vector=True
)

print(f'HOG feature vector size: {hog_feat_m.shape[0]}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('HOG brushstroke orientation maps\n'
             f'Cell size={HOG_CELL}px — each bar = dominant gradient direction in cell', fontsize=12)

for col, (name, img, vis) in enumerate([
    ('Manet', img_m, hog_vis_m), ('Contemporary', img_c, hog_vis_c)
]):
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'{name}')
    axes[0, col].axis('off')

    axes[1, col].imshow(vis, cmap='gray')
    axes[1, col].set_title(f'{name} — HOG visualisation')
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

---
## 4. Combined Feature Vector + PCA Scatter

Concatenate [LBP histogram | HOG feature vector] and apply PCA to 2D. With only 2 paintings this just shows the direction of maximum variation — **run with more images to see real clustering**.

In [ ]:
def extract_lbp_hog_features(gray: np.ndarray,
                              lbp_p: int = 8, lbp_r: int = 1,
                              hog_cell: int = 16) -> np.ndarray:
    """Extract combined LBP^{riu2} + HOG feature vector."""
    # LBP histogram
    lbp = local_binary_pattern(gray, P=lbp_p, R=lbp_r, method='uniform')
    lbp_hist = lbp_histogram(lbp)

    # HOG feature vector
    hog_feat = hog(gray, orientations=9,
                   pixels_per_cell=(hog_cell, hog_cell),
                   cells_per_block=(2, 2),
                   block_norm='L2-Hys', feature_vector=True)

    return np.concatenate([lbp_hist, hog_feat])


feat_m = extract_lbp_hog_features(gray_m)
feat_c = extract_lbp_hog_features(gray_c)

print(f'Combined feature vector size: {feat_m.shape[0]}')
print(f'  LBP part:  59 features')
print(f'  HOG part:  {feat_m.shape[0] - 59} features')

# L2 distance
feat_m_norm = feat_m / (np.linalg.norm(feat_m) + 1e-9)
feat_c_norm = feat_c / (np.linalg.norm(feat_c) + 1e-9)
print(f'\nCosine similarity (LBP+HOG): {np.dot(feat_m_norm, feat_c_norm):.4f}')
print(f'L2 distance:                 {np.linalg.norm(feat_m_norm - feat_c_norm):.4f}')

In [ ]:
# Visualise feature vector alignment
# Use only the LBP part (59 features) for clear visualisation

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(feat_m_norm[:59], color='#2196F3', lw=1.5, label='Manet (LBP part)')
ax.plot(feat_c_norm[:59], color='#FF5722', lw=1.5, alpha=0.8, label='Contemporary (LBP part)')
ax.fill_between(range(59), feat_m_norm[:59], feat_c_norm[:59], alpha=0.2, color='grey',
                label='Difference')
ax.set_xlabel('LBP code (0–58)')
ax.set_ylabel('Normalised frequency')
ax.set_title('LBP histogram overlap — filled area = discriminative region')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\nWith more paintings: run PCA on the full feature matrix to see 2D clustering.')
print('Example (requires N > 2 images):')
print('  X = np.stack([feat_1, feat_2, ..., feat_N])  # (N, n_features)')
print('  pca = PCA(n_components=2).fit(X)')
print('  X_2d = pca.transform(X)')
print('  plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels)')

---
## 5. Key Observations

| Question | Observation |
|---|---|
| Which LBP codes are most different between Manet and contemporary? | |
| Does code 58 (non-uniform) differ? (indicates irregular texture) | |
| HOG: do orientation patterns look different visually? | |
| L2 distance LBP+HOG: is it large relative to self-similarity? | |
| Overall: useful as Tier 2 complement? | |